# Notebook 5: GA-Guided De Novo Design (Google Colab)
**Platform:** Google Colab (CPU runtime)
**Input:** Upload `X_train_ecfp4.npy`, `X_train_desc.csv`, `y_train.npy`, `train.csv` when prompted
**Output:** `novel_antifungal_candidates.csv`, `all_GA_results.csv`, `GA_results.png`
**Runtime estimate:** 30-60 minutes on Colab CPU


In [ ]:
# Install dependencies
import subprocess
for pkg in ['selfies','xgboost','rdkit-pypi']:
    subprocess.run(['pip','install','-q',pkg])
print('Dependencies ready.')

In [ ]:
# Upload required files
from google.colab import files
print('Upload: X_train_ecfp4.npy, X_train_desc.csv, y_train.npy, train.csv')
uploaded = files.upload()
DATA_DIR = '/content'
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
import pandas as pd, numpy as np, random, warnings, os
warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.DataStructs import ConvertToNumpyArray
from xgboost import XGBClassifier

SEED = 42; random.seed(SEED); np.random.seed(SEED)

X_train_fp   = np.load(os.path.join(DATA_DIR,'X_train_ecfp4.npy'))
X_train_desc = pd.read_csv(os.path.join(DATA_DIR,'X_train_desc.csv'))
y_train      = np.load(os.path.join(DATA_DIR,'y_train.npy'))
X_train_comb = np.hstack([X_train_fp, X_train_desc.values])
desc_names   = list(X_train_desc.columns)
train_desc_medians = X_train_desc.median()
df_train     = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))

scale_pos = (y_train==0).sum()/y_train.sum()
model = XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,
    subsample=0.8,colsample_bytree=0.8,scale_pos_weight=scale_pos,
    eval_metric='logloss',random_state=SEED,n_jobs=-1)
model.fit(X_train_comb,y_train)
print('Model retrained.')

In [ ]:
# Rebuild AD
def get_fp(smi,radius=2,nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(mol,radius,nBits=nbits)

train_fps = [get_fp(s) for s in df_train['std_smiles']]
train_fps = [f for f in train_fps if f is not None]
K = 5
sample_fps = train_fps[:500]
self_sims  = []
for i,fp in enumerate(sample_fps):
    others = sample_fps[:i]+sample_fps[i+1:]
    sims   = sorted(DataStructs.BulkTanimotoSimilarity(fp,others),reverse=True)
    self_sims.append(np.mean(sims[:K]))
self_sims    = np.array(self_sims)
AD_THRESHOLD = self_sims.mean()-0.5*self_sims.std()
print(f'AD threshold: {AD_THRESHOLD:.4f}')

In [ ]:
# Scoring + drug-likeness filter
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

def score_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return 0.0,0,0.0
        fp  = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
        arr = np.zeros(2048,dtype=np.uint8); ConvertToNumpyArray(fp,arr)
        desc = list(calculator.CalcDescriptors(mol))
        desc = [train_desc_medians[i] if (v!=v or abs(v)>1e10) else v for i,v in enumerate(desc)]
        X    = np.hstack([arr,desc]).reshape(1,-1)
        prob = float(model.predict_proba(X)[0,1])
        sims = sorted(DataStructs.BulkTanimotoSimilarity(fp,train_fps),reverse=True)
        knn  = float(np.mean(sims[:K]))
        return prob,(1 if knn>=AD_THRESHOLD else 0),knn
    except: return 0.0,0,0.0

def fitness(smi,ad_penalty=0.3):
    prob,in_ad,_ = score_smiles(smi)
    return prob*(1-ad_penalty) if in_ad==0 else prob

def is_druglike(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return False
        return (200<=Descriptors.MolWt(mol)<=600 and
                -1<=Descriptors.MolLogP(mol)<=6 and
                rdMolDescriptors.CalcNumHBD(mol)<=5 and
                rdMolDescriptors.CalcNumHBA(mol)<=10 and
                rdMolDescriptors.CalcNumRotatableBonds(mol)<=10 and
                mol.GetNumHeavyAtoms()>=15)
    except: return False

# Test on fluconazole
p,ad,k = score_smiles('OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F')
print(f'Fluconazole: prob={p:.4f}, AD={ad}, kNN={k:.4f}')

In [ ]:
# Substituent library and core templates
ALL_SUBS = ['F','Cl','Br','C','CC','CCC','C(C)C','OC','OCC','OC(C)C','O',
            'C(F)(F)F','C#N','S(=O)(=O)N','C(=O)N','C(=O)OC','N(=O)=O',
            'n1ccnc1','n1cccc1','c1ccncc1','c1cncnc1','c1ccoc1','c1ccsc1',
            'N','NC','NCC','NC(C)C','NC(=O)C']

TEMPLATES = [
    'OC(Cn1cncn1)(Cn1cncn1)c1ccc({R1})cc1',
    'OC(Cn1cncn1)(Cn1cncn1)c1cc({R1})cc(F)c1',
    'OC(Cn1cncn1)(Cn1cncn1)c1ccc(CC{R1})cc1',
    'OC(Cn1cncn1)(Cc1ncc({R1})cn1)c1ccc(F)cc1',
    'OC(Cn1cncn1)(Cn1cncn1)c1cc(F)c({R1})cc1F',
]

def build_mol(template,r1):
    smi = template.replace('{R1}',r1)
    try:
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol,canonical=True) if mol else None
    except: return None

print(f'Substituents: {len(ALL_SUBS)} | Templates: {len(TEMPLATES)}')

In [ ]:
# Initial population
initial_pop = set()
for t in TEMPLATES:
    for s in ALL_SUBS:
        smi = build_mol(t,s)
        if smi and is_druglike(smi): initial_pop.add(smi)

# Add known actives as seeds
for s in ['OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F','OC(Cn1cncn1)(Cn1cncn1)c1ccc(Cl)cc1',
          'CC[C@@H](Cn1cncn1)[C@@](O)(c1ccc(F)cc1F)c1ncc(F)cn1']:
    mol = Chem.MolFromSmiles(s)
    if mol: initial_pop.add(Chem.MolToSmiles(mol))

initial_pop  = list(initial_pop)
print(f'Initial population: {len(initial_pop)}')
init_scores  = [(s,fitness(s)) for s in initial_pop]
init_scores.sort(key=lambda x:x[1],reverse=True)
print('Top 5 initial scores:')
for s,sc in init_scores[:5]:
    p,ad,k = score_smiles(s)
    print(f'  score={sc:.4f} prob={p:.4f} AD={ad} {s}')

In [ ]:
# Genetic algorithm
def tournament_select(scored,k=3):
    cands = random.sample(scored,min(k,len(scored)))
    return max(cands,key=lambda x:x[1])[0]

def mutate(smi,templates,subs,rate=0.4):
    if random.random()>rate: return smi
    new = build_mol(random.choice(templates),random.choice(subs))
    return new if new and is_druglike(new) else smi

POP_SIZE, N_GEN, N_ELITE = 150, 60, 15
population    = [s for s,_ in init_scores[:POP_SIZE]]
while len(population)<POP_SIZE:
    s = build_mol(random.choice(TEMPLATES),random.choice(ALL_SUBS))
    if s and is_druglike(s) and s not in population: population.append(s)

all_evaluated  = {}
best_per_gen   = []
print(f'GA: pop={POP_SIZE}, generations={N_GEN}')
for gen in range(N_GEN):
    scored = []
    for s in population:
        if s not in all_evaluated: all_evaluated[s] = fitness(s)
        scored.append((s,all_evaluated[s]))
    scored.sort(key=lambda x:x[1],reverse=True)
    best_per_gen.append((gen,scored[0][0],scored[0][1]))
    if gen%10==0:
        p,ad,k = score_smiles(scored[0][0])
        print(f'Gen {gen:3d} | best={scored[0][1]:.4f} prob={p:.4f} AD={ad}')
    new_pop = [s for s,_ in scored[:N_ELITE]]
    while len(new_pop)<POP_SIZE:
        p1 = tournament_select(scored)
        child = mutate(random.choice([p1,tournament_select(scored)]),TEMPLATES,ALL_SUBS)
        new_pop.append(child if child and is_druglike(child) else scored[random.randint(0,19)][0])
    population = new_pop
print(f'GA complete. Evaluated: {len(all_evaluated)} unique compounds.')

In [ ]:
# Extract and filter results
results = []
for smi,fit in all_evaluated.items():
    p,ad,k = score_smiles(smi)
    mol = Chem.MolFromSmiles(smi)
    if mol is None: continue
    results.append({'smiles':smi,'prob_active':round(p,4),'fitness':round(fit,4),
                    'within_AD':ad,'knn_sim':round(k,4),
                    'MW':round(Descriptors.MolWt(mol),2),
                    'LogP':round(Descriptors.MolLogP(mol),3),
                    'HBD':rdMolDescriptors.CalcNumHBD(mol),
                    'HBA':rdMolDescriptors.CalcNumHBA(mol)})
df_results = pd.DataFrame(results).sort_values('prob_active',ascending=False).drop_duplicates('smiles').reset_index(drop=True)

known = {'OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F'}
df_novel = df_results[(df_results['within_AD']==1)&(df_results['prob_active']>=0.7)&(~df_results['smiles'].isin(known))].reset_index(drop=True)

# Diversity filter
def diversity_filter(df,thr=0.85):
    sel_fps,sel_idx = [],[]
    for idx,row in df.iterrows():
        fp = get_fp(row['smiles'])
        if fp is None: continue
        if not sel_fps: sel_fps.append(fp); sel_idx.append(idx); continue
        if max(DataStructs.BulkTanimotoSimilarity(fp,sel_fps))<thr:
            sel_fps.append(fp); sel_idx.append(idx)
    return df.loc[sel_idx].reset_index(drop=True)

df_diverse = diversity_filter(df_novel)
print(f'Diverse novel candidates (prob>=0.7, within AD, Tanimoto<0.85): {len(df_diverse)}')
print(df_diverse[['smiles','prob_active','knn_sim','MW','LogP','HBD']].to_string(index=False))

In [ ]:
# Visualisations
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
fig,axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('GA-Generated Novel Antifungal Candidates',fontsize=13,fontweight='bold')
axes[0,0].plot([g for g,_,_ in best_per_gen],[f for _,_,f in best_per_gen],color='#2ecc71',linewidth=2)
axes[0,0].set_xlabel('Generation'); axes[0,0].set_ylabel('Best fitness')
axes[0,0].set_title('GA Convergence'); axes[0,0].set_ylim(0,1); axes[0,0].grid(alpha=0.3)
axes[0,1].hist(df_results['prob_active'],bins=30,color='steelblue',edgecolor='white',alpha=0.85)
axes[0,1].axvline(0.7,color='red',linestyle='--',linewidth=1.2,label='Threshold=0.7')
axes[0,1].set_xlabel('Predicted probability'); axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Probability Distribution'); axes[0,1].legend()
sc = axes[1,0].scatter(df_diverse['MW'],df_diverse['LogP'],c=df_diverse['prob_active'],
    cmap='RdYlGn',s=40,alpha=0.8,vmin=0.5,vmax=1.0)
plt.colorbar(sc,ax=axes[1,0],label='Prob(active)')
axes[1,0].set_xlabel('MW'); axes[1,0].set_ylabel('LogP'); axes[1,0].set_title('Chemical Space - Novel Candidates')
axes[1,1].hist(df_diverse['knn_sim'],bins=12,color='#e67e22',edgecolor='white',alpha=0.85)
axes[1,1].axvline(AD_THRESHOLD,color='black',linestyle='--',linewidth=1.2,label=f'AD threshold={AD_THRESHOLD:.3f}')
axes[1,1].set_xlabel('Mean kNN Tanimoto'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title('AD Similarity'); axes[1,1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('/content/GA_results.png',dpi=150,bbox_inches='tight'); plt.show()

df_diverse.to_csv('/content/novel_antifungal_candidates.csv',index=False)
df_results.to_csv('/content/all_GA_results.csv',index=False)
from google.colab import files
files.download('/content/novel_antifungal_candidates.csv')
files.download('/content/all_GA_results.csv')
files.download('/content/GA_results.png')
print(f'Done. {len(df_diverse)} novel candidates generated.')